# Career Skills ETL — PySpark on Colab
**Project:** `veerababu33` | **Bucket:** `gs://veerababu33-career-etl/` | **BQ Dataset:** `career_analytics`

## Cell 1 · Install Java + PySpark

In [2]:
import subprocess, os

# Check if Java already installed
java_check = os.popen("java -version 2>&1").read()
if "version" not in java_check:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-11-jdk-headless"],
                   check=True)

# Find Java home
java_home = os.popen("dirname $(dirname $(readlink -f $(which java)))").read().strip()
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ["PATH"]

subprocess.run(["pip", "install", "pyspark==3.5.1", "db-dtypes", "--quiet"], check=True)

import pyspark
print(f"PySpark {pyspark.__version__} ready")
print(f"Java: {os.popen('java -version 2>&1').read().splitlines()[0]}")
print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

PySpark 3.5.1 ready
Java: openjdk version "17.0.17" 2025-10-21
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


## Cell 2 · GCP Auth

In [4]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated ✓")


Authenticated ✓


## Cell 3 · Config

In [5]:
import os

GCS_BUCKET  = "veerababu33-career-etl"
GCP_PROJECT = "veerababu33"
BQ_DATASET  = "career_analytics"

GCS_PROCESSED = f"gs://{GCS_BUCKET}/processed"

BQ_TABLES = {
    "skills_metrics":     f"{GCP_PROJECT}.{BQ_DATASET}.skills_metrics",
    "occupation_metrics": f"{GCP_PROJECT}.{BQ_DATASET}.occupation_metrics",
    "occupations_clean":  f"{GCP_PROJECT}.{BQ_DATASET}.occupations_clean",
}

IMPORTANCE_SCALE_ID = "IM"
WAGES_GROUP_FILTER  = "detailed"
MIN_SKILL_OCC_COUNT = 5

LOCAL_DATA   = "/content/data"
LOCAL_OUTPUT = "/content/output"
os.makedirs(LOCAL_DATA,   exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

print(f"Bucket  : gs://{GCS_BUCKET}")
print(f"Project : {GCP_PROJECT}")
print(f"BQ      : {GCP_PROJECT}.{BQ_DATASET}")


Bucket  : gs://veerababu33-career-etl
Project : veerababu33
BQ      : veerababu33.career_analytics


## Cell 4 · Upload CSVs
Upload: `skills_clean.csv`, `occupations_clean.csv`, `education_preferred.csv`, `wages_clean.csv`

In [8]:
from google.colab import files

uploaded = files.upload()

for fname, content in uploaded.items():
    with open(f"{LOCAL_DATA}/{fname}", "wb") as f:
        f.write(content)
    print(f"Saved → {LOCAL_DATA}/{fname}")

required = {"skills_clean.csv", "occupations_clean.csv",
            "education_preferred.csv", "wages_clean.csv"}
missing = required - set(uploaded.keys())
if missing:
    raise ValueError(f"Missing: {missing}")
print("All files ready ✓")


Saving education_preferred.csv to education_preferred.csv
Saving occupations_clean.csv to occupations_clean.csv
Saving skills_clean.csv to skills_clean.csv
Saving wages_clean.csv to wages_clean.csv
Saved → /content/data/education_preferred.csv
Saved → /content/data/occupations_clean.csv
Saved → /content/data/skills_clean.csv
Saved → /content/data/wages_clean.csv
All files ready ✓


## Cell 5 · Start SparkSession

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CareerSkillsETL")
    .master("local[*]")
    .config("spark.driver.memory",          "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled",   "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version} on {spark.sparkContext.master}")
print(f"Cores: {spark.sparkContext.defaultParallelism}")


Spark 3.5.1 on local[*]
Cores: 2


## Cell 6 · Load CSVs into Spark

In [10]:
from pyspark.sql.types import *

SKILLS_SCHEMA = StructType([
    StructField("soc_code",      StringType(), True),
    StructField("title",         StringType(), True),
    StructField("element_id",    StringType(), True),
    StructField("element_name",  StringType(), True),
    StructField("scale_id",      StringType(), True),
    StructField("scale_name",    StringType(), True),
    StructField("data_value",    DoubleType(), True),
    StructField("date",          StringType(), True),
    StructField("domain_source", StringType(), True),
])
OCCS_SCHEMA = StructType([
    StructField("soc_code",    StringType(), True),
    StructField("title",       StringType(), True),
    StructField("description", StringType(), True),
])
EDU_SCHEMA = StructType([
    StructField("soc_code",               StringType(), True),
    StructField("preferred_edu_category", StringType(), True),
    StructField("preferred_edu_pct",      DoubleType(), True),
    StructField("preferred_education",    StringType(), True),
])
WAGES_SCHEMA = StructType([
    StructField("soc_code",  StringType(), True),
    StructField("occ_title", StringType(), True),
    StructField("o_group",   StringType(), True),
    StructField("tot_emp",   DoubleType(), True),
    StructField("h_mean",    DoubleType(), True),
    StructField("a_mean",    DoubleType(), True),
    StructField("h_median",  DoubleType(), True),
    StructField("a_median",  DoubleType(), True),
])

def read_csv(fname, schema):
    return spark.read.option("header","true").schema(schema).csv(f"{LOCAL_DATA}/{fname}")

skills_raw = read_csv("skills_clean.csv",        SKILLS_SCHEMA)
occs_raw   = read_csv("occupations_clean.csv",   OCCS_SCHEMA)
edu_raw    = read_csv("education_preferred.csv",  EDU_SCHEMA)
wages_raw  = read_csv("wages_clean.csv",          WAGES_SCHEMA)

print(f"skills      : {skills_raw.count():,}")
print(f"occupations : {occs_raw.count():,}")
print(f"education   : {edu_raw.count():,}")
print(f"wages       : {wages_raw.count():,}")


skills      : 62,580
occupations : 1,016
education   : 873
wages       : 1,403


## Cell 7 · Clean & Filter

In [11]:
from pyspark.sql import functions as F

# Skills — IM scale only, valid range
skills_im = (
    skills_raw
    .withColumn("soc_code",     F.trim(F.col("soc_code")))
    .withColumn("element_name", F.trim(F.col("element_name")))
    .withColumn("scale_id",     F.trim(F.upper(F.col("scale_id"))))
    .dropna(subset=["soc_code","element_name","data_value"])
    .filter(F.col("scale_id") == IMPORTANCE_SCALE_ID)
    .filter(F.col("data_value").between(0, 6))
    .cache()
)
print(f"skills_im  : {skills_im.count():,}  (from {skills_raw.count():,} — IM only)")

# Wages — detailed SOC rows only (drop total/major/minor/broad rollups)
wages_det = (
    wages_raw
    .withColumn("soc_code", F.trim(F.col("soc_code")))
    .filter(F.col("o_group") == WAGES_GROUP_FILTER)
    .dropna(subset=["soc_code","a_median","tot_emp"])
    .filter(F.col("a_median") > 0)
    .filter(F.col("tot_emp")  > 0)
    .cache()
)
print(f"wages_det  : {wages_det.count():,}  (from {wages_raw.count():,} — detailed only)")

# Occupations
occs = (
    occs_raw
    .withColumn("soc_code", F.trim(F.col("soc_code")))
    .dropna(subset=["soc_code","title"])
    .dropDuplicates(["soc_code"])
)
print(f"occs       : {occs.count():,}")

# Education
edu_pref = (
    edu_raw
    .withColumn("soc_code", F.trim(F.col("soc_code")))
    .dropna(subset=["soc_code","preferred_education"])
    .dropDuplicates(["soc_code"])
)
print(f"edu_pref   : {edu_pref.count():,}")


skills_im  : 31,290  (from 62,580 — IM only)
wages_det  : 809  (from 1,403 — detailed only)
occs       : 1,016
edu_pref   : 873


## Cell 8 · Merge Datasets

In [12]:
base = (
    skills_im
    .join(wages_det.select("soc_code","tot_emp","a_mean","a_median"), on="soc_code", how="inner")
    .join(edu_pref.select("soc_code","preferred_education","preferred_edu_pct"), on="soc_code", how="left")
    .join(occs.select("soc_code", F.col("description").alias("occ_description")), on="soc_code", how="left")
    .cache()
)

print(f"Base joined  : {base.count():,} rows")
print(f"Occupations  : {base.select('soc_code').distinct().count():,}")
print(f"Skills       : {base.select('element_name').distinct().count():,}")
base.show(3, truncate=50)


Base joined  : 24,850 rows
Occupations  : 710
Skills       : 35
+--------+----------------+----------+---------------------+--------+----------+----------+-------+-------------+--------+--------+--------+-------------------+-----------------+--------------------------------------------------+
|soc_code|           title|element_id|         element_name|scale_id|scale_name|data_value|   date|domain_source| tot_emp|  a_mean|a_median|preferred_education|preferred_edu_pct|                                   occ_description|
+--------+----------------+----------+---------------------+--------+----------+----------+-------+-------------+--------+--------+--------+-------------------+-----------------+--------------------------------------------------+
| 11-1011|Chief Executives|   2.A.1.a|Reading Comprehension|      IM|Importance|      4.12|08/2023|      Analyst|211850.0|262930.0|206420.0|    Master's Degree|            45.91|Determine and formulate policies and provide ov...|
| 11-1011|Chief 

## Cell 9 · Compute Metrics

In [13]:
from pyspark.sql.window import Window

def min_max_norm(df, raw_col, out_col):
    s = df.agg(F.min(raw_col).alias("mn"), F.max(raw_col).alias("mx")).collect()[0]
    if s["mx"] == s["mn"]:
        return df.withColumn(out_col, F.lit(0.0))
    return df.withColumn(out_col, F.round((F.col(raw_col) - s["mn"]) / (s["mx"] - s["mn"]), 4))

# ── skills_metrics ────────────────────────────────────────────────────────────
# Skill Demand Score  = employment-weighted avg importance, normalized [0,1]
# Salary Impact Score = importance-weighted avg median salary, normalized [0,1]

sk_agg = (
    base.groupBy("element_name")
    .agg(
        F.countDistinct("soc_code").alias("occ_count"),
        F.round(F.avg("data_value"), 4).alias("avg_importance"),
        F.round(F.max("data_value"), 4).alias("max_importance"),
        F.round(F.avg("a_median"),   2).alias("avg_occ_salary"),
        (F.sum(F.col("data_value") * F.col("tot_emp")) / F.sum("tot_emp")).alias("_demand_raw"),
        (F.sum(F.col("data_value") * F.col("a_median")) / F.sum("data_value")).alias("_salary_raw"),
    )
    .filter(F.col("occ_count") >= MIN_SKILL_OCC_COUNT)
)
sk_agg = min_max_norm(sk_agg, "_demand_raw", "skill_demand_score")
sk_agg = min_max_norm(sk_agg, "_salary_raw", "salary_impact_score")
skills_metrics = sk_agg.drop("_demand_raw","_salary_raw").orderBy(F.col("skill_demand_score").desc()).cache()
print(f"skills_metrics: {skills_metrics.count()} skills")
skills_metrics.show(10, truncate=30)

# ── occupation_metrics ────────────────────────────────────────────────────────
w = Window.partitionBy("soc_code").orderBy(F.col("data_value").desc())
top_skill = (
    base.withColumn("_r", F.row_number().over(w)).filter(F.col("_r")==1)
    .select("soc_code", F.col("element_name").alias("top_skill"))
)
occ_agg = (
    base.groupBy("soc_code","title")
    .agg(
        F.round(F.avg("data_value"),4).alias("avg_skill_importance"),
        F.countDistinct("element_name").alias("skill_count"),
        F.first("tot_emp").alias("tot_emp"),
        F.first("a_mean").alias("a_mean"),
        F.first("a_median").alias("a_median"),
        F.first("preferred_education").alias("preferred_education"),
        F.first("preferred_edu_pct").alias("preferred_edu_pct"),
        F.first("occ_description").alias("description"),
    )
)
occupation_metrics = occ_agg.join(top_skill, on="soc_code", how="left").orderBy(F.col("a_median").desc()).cache()
print(f"occupation_metrics: {occupation_metrics.count()} occupations")
occupation_metrics.select("soc_code","title","a_median","avg_skill_importance","top_skill").show(10, truncate=40)

# ── occupations_clean ─────────────────────────────────────────────────────────
occupations_clean = (
    occs
    .join(wages_det.select("soc_code","tot_emp","a_mean","a_median","h_median"), on="soc_code", how="left")
    .join(edu_pref.select("soc_code","preferred_education","preferred_edu_pct"), on="soc_code", how="left")
    .orderBy("soc_code").cache()
)
print(f"occupations_clean: {occupations_clean.count()} rows")


skills_metrics: 35 skills
+----------------------------+---------+--------------+--------------+--------------+------------------+-------------------+
|                element_name|occ_count|avg_importance|max_importance|avg_occ_salary|skill_demand_score|salary_impact_score|
+----------------------------+---------+--------------+--------------+--------------+------------------+-------------------+
|            Active Listening|      710|        3.5448|           5.0|      67128.76|               1.0|             0.4821|
|                    Speaking|      710|        3.4925|          4.88|      67128.76|            0.9742|             0.4887|
|           Critical Thinking|      710|        3.4497|          4.88|      67128.76|            0.9081|             0.5474|
|       Reading Comprehension|      710|        3.4037|          4.88|      67128.76|            0.9077|             0.5703|
|                  Monitoring|      710|        3.2644|          4.25|      67128.76|            0.

## Cell 10 · Write Parquet to GCS

In [15]:
import glob

def write_parquet_gcs(df, gcs_path):
    local_path = "/content/output/" + gcs_path.split("/")[-1]
    (df.coalesce(1).write.mode("overwrite")
     .option("compression","snappy").parquet(local_path))
    parts = glob.glob(f"{local_path}/part-*.parquet")
    from google.cloud import storage
    client = storage.Client(project=GCP_PROJECT)
    bucket = client.bucket(GCS_BUCKET)
    for p in parts:
        blob_name = gcs_path.replace(f"gs://{GCS_BUCKET}/","") + "/" + p.split("/")[-1]
        bucket.blob(blob_name).upload_from_filename(p)
        print(f"  Uploaded → gs://{GCS_BUCKET}/{blob_name}")

print("Writing to GCS...")
write_parquet_gcs(skills_metrics,     f"{GCS_PROCESSED}/skills_metrics")
write_parquet_gcs(occupation_metrics, f"{GCS_PROCESSED}/occupation_metrics")
write_parquet_gcs(occupations_clean,  f"{GCS_PROCESSED}/occupations_clean")
print("Done ✓")


Writing to GCS...
  Uploaded → gs://veerababu33-career-etl/processed/skills_metrics/part-00000-caa768bd-875a-4ddc-a677-b1194ea56599-c000.snappy.parquet
  Uploaded → gs://veerababu33-career-etl/processed/occupation_metrics/part-00000-ed7859f1-870a-4ce5-a933-332da4f65f83-c000.snappy.parquet
  Uploaded → gs://veerababu33-career-etl/processed/occupations_clean/part-00000-d84d0c99-a07d-4dbc-8eab-2538005f1008-c000.snappy.parquet
Done ✓


## Cell 11 · Load to BigQuery

In [17]:
from google.cloud import bigquery

bq = bigquery.Client(project=GCP_PROJECT)
bq.create_dataset(bigquery.Dataset(f"{GCP_PROJECT}.{BQ_DATASET}"), exists_ok=True)

def load_bq(spark_df, table_id):
    pdf = spark_df.toPandas()
    job = bq.load_table_from_dataframe(
        pdf, table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            autodetect=True,
        )
    )
    job.result()
    n = bq.get_table(table_id).num_rows
    print(f"  {table_id}  →  {n:,} rows")

print("Loading to BigQuery...")
load_bq(skills_metrics,     BQ_TABLES["skills_metrics"])
load_bq(occupation_metrics, BQ_TABLES["occupation_metrics"])
load_bq(occupations_clean,  BQ_TABLES["occupations_clean"])
print("All tables loaded ✓")


Loading to BigQuery...
  veerababu33.career_analytics.skills_metrics  →  35 rows
  veerababu33.career_analytics.occupation_metrics  →  710 rows
  veerababu33.career_analytics.occupations_clean  →  1,016 rows
All tables loaded ✓


## Cell 12 · Verify

In [18]:
print("BigQuery row counts:")
for name, table_id in BQ_TABLES.items():
    n = bq.query(f"SELECT COUNT(*) AS n FROM `{table_id}`").to_dataframe()["n"].iloc[0]
    print(f"  {name:<25} {n:>6,} rows")

print("\nGCS Parquet files:")
from google.cloud import storage
client = storage.Client(project=GCP_PROJECT)
for blob in client.list_blobs(GCS_BUCKET, prefix="processed/"):
    if blob.name.endswith(".parquet"):
        print(f"  gs://{GCS_BUCKET}/{blob.name}  ({blob.size/1024:.0f} KB)")


BigQuery row counts:
  skills_metrics                35 rows
  occupation_metrics           710 rows
  occupations_clean          1,016 rows

GCS Parquet files:
  gs://veerababu33-career-etl/processed/occupation_metrics/part-00000-ed7859f1-870a-4ce5-a933-332da4f65f83-c000.snappy.parquet  (125 KB)
  gs://veerababu33-career-etl/processed/occupations_clean/part-00000-d84d0c99-a07d-4dbc-8eab-2538005f1008-c000.snappy.parquet  (151 KB)
  gs://veerababu33-career-etl/processed/skills_metrics/part-00000-caa768bd-875a-4ddc-a677-b1194ea56599-c000.snappy.parquet  (4 KB)


In [21]:
from google.colab import files

skills_metrics.toPandas().to_csv("skills_metrics.csv", index=False)
occupation_metrics.toPandas().to_csv("occupation_metrics.csv", index=False)
occupations_clean.toPandas().to_csv("occupations_clean.csv", index=False)

files.download("skills_metrics.csv")
files.download("occupation_metrics.csv")
files.download("occupations_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>